In [7]:
# packages and working directory  
import scipy 
import sklearn
import econml 
import arch
import os 
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib as plt
import statsmodels.api as sm 
from statsmodels.discrete.discrete_model import Probit
from statsmodels.iolib.summary2 import summary_col
import statsmodels.formula.api as smf 
from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt 
from scipy import stats
from scipy.stats import ttest_ind
from scipy.optimize import approx_fprime

from torch_choice.data import ChoiceDataset
from torch_choice.model import ConditionalLogitModel
# Consolidate changing directory and CPI dictionary since these don't change throughout the script
new_directory = r'C:\Users\hisham\Spain\2021 datasets'
os.chdir(new_directory)



In [14]:
import torch
from torch_choice.data import ChoiceDataset
from torch_choice.model import ConditionalLogitModel
from torch.utils.data import DataLoader

In [33]:
df = pd.read_csv('manychoicestalong.csv')

# Define the list of scenarios
scenarios = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

# Create the 'labor' column based on the 'scenario' column
df['labor'] = df.apply(lambda row: row[f'lhw_{int(row["scenario"])}'], axis=1)

df['log_y'] = np.log(df['ils_udb_yds'])
df['log_l'] = np.log(80 - df['labor'])
df['log2_y'] = 0.5* df['log_y']**2 
df['log2_l'] = 0.5* df['log_l']**2
df['log_y_l'] = df['log_y'] * df['log_l']



In [3]:
# Assume df is your DataFrame and it's already prepared with the necessary log transformations
# Ensure that 'scenario' is categorical and 'idperson' is set for groups
df['scenario'] = df['scenario'].astype('category')
df['choice_made'] = df['choice_made'].astype('category')

# Creating the independent variables matrix including intercept
X = df[['log_y', 'log2_y', 'log_l', 'log2_l', 'log_y_l']]
X = sm.add_constant(X)  # Adds a constant term for the intercept

# Dependent variable
y = df['choice_made']

# Fit the model
model = sm.MNLogit(y, X, groups=df['idperson'])
result = model.fit(method='bfgs', maxiter=1000)  # You can adjust the optimization method and iterations

print(result.summary())


c:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:130: ValueWarning: unknown kwargs ['groups']
  warnings.warn(msg, ValueWarning)


Optimization terminated successfully.
         Current function value: 0.201829
         Iterations: 78
         Function evaluations: 81
         Gradient evaluations: 81


c:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:130: ValueWarning: unknown kwargs ['groups']
  warnings.warn(msg, ValueWarning)


                          MNLogit Regression Results                          
Dep. Variable:            choice_made   No. Observations:                68704
Model:                        MNLogit   Df Residuals:                    68698
Method:                           MLE   Df Model:                            5
Date:                Fri, 12 Apr 2024   Pseudo R-squ.:                  0.1367
Time:                        10:37:16   Log-Likelihood:                -13866.
converged:                       True   LL-Null:                       -16062.
Covariance Type:            nonrobust   LLR p-value:                     0.000
choice_made=1       coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const           -76.8465      5.942    -12.932      0.000     -88.493     -65.200
log_y             0.9193      0.916      1.004      0.315      -0.876       2.714
log2_y           -0.2413      0.085     

In [28]:
from time import time
import pandas as pd
import torch

from torch_choice.data import ChoiceDataset, utils
from torch_choice.model import ConditionalLogitModel

from torch_choice import run

In [29]:
if torch.cuda.is_available():
    print(f'CUDA device used: {torch.cuda.get_device_name()}')
    device = 'cuda'
else:
    print('Running tutorial on CPU.')
    device = 'cpu'

Running tutorial on CPU.


In [30]:

df.sort_values(by='idperson', inplace=True)
df.head()

,idhh,idperson,idmother,idfather,idpartner,idorighh,idorigperson,dag,dgn,dec,...,lhw_13,lhw_14,lhw_15,choice_made,labor,log_y,log_l,log2_y,log2_l,log_y_l
0,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,64.0,69.0,73.0,0,0.0,6.715868,4.382027,22.551442,9.601079,29.429113
60116,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,64.0,69.0,73.0,0,69.0,8.020826,2.397895,32.166824,2.874951,19.233100
55822,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,64.0,69.0,73.0,0,64.0,7.959901,2.772589,31.680014,3.843624,22.069532
51528,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,64.0,69.0,73.0,0,58.0,7.879556,3.091042,31.043705,4.777272,24.356043
47234,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,64.0,69.0,73.0,0,54.0,7.818157,3.258097,30.561787,5.307597,25.472309


In [34]:
df.shape

(68704, 367)

In [36]:
item_index = df[df['choice_made'] == 1].sort_values(by='idperson')['scenario'].reset_index(drop=True)
print(item_index)

0        9
1        9
2        0
3        8
4        8
        ..
4289    10
4290     0
4291     8
4292     9
4293    10
Name: scenario, Length: 4294, dtype: int64


In [41]:
item_index.value_counts()

scenario
8     2345
7      336
10     269
9      230
0      199
6      198
4      194
12     180
5      132
2       67
3       61
11      37
1       18
14      17
13       7
15       4
Name: count, dtype: int64

In [45]:

item_index = df[df['choice_made'] == 1].sort_values(by='idperson')['scenario'].reset_index(drop=True)
print(item_index)

item_names = list(range(16))  # This includes all scenarios from 0 to 15
num_items = 16

# Creating an encoder that maps each scenario directly to an index (identity mapping in this case)
encoder = {name: name for name in item_names}
print(f"{encoder=:}")

# Map the item_index using the corrected encoder
item_index = item_index.map(encoder)  # This should now work without throwing a KeyError
item_index = torch.LongTensor(item_index)
print(f"{item_index=:}")


0        9
1        9
2        0
3        8
4        8
        ..
4289    10
4290     0
4291     8
4292     9
4293    10
Name: scenario, Length: 4294, dtype: int64
encoder={0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15}
item_index=tensor([ 9,  9,  0,  ...,  8,  9, 10])


In [48]:
price_cost_freq_ovt = utils.pivot3d(df, dim0='idperson', dim1='scenario',
                                    values=['yivwg','dag' , 'dgn'])
print(f'{price_cost_freq_ovt.shape=:}')

price_ivt = utils.pivot3d(df, dim0='idperson', dim1='scenario', values='ils_udb_yds')
print(f'{price_ivt.shape=:}')

price_cost_freq_ovt.shape=torch.Size([4294, 16, 3])
price_ivt.shape=torch.Size([4294, 16, 1])


In [ ]:
dataset = ChoiceDataset(item_index=item_index,
                        price_cost_freq_ovt=price_cost_freq_ovt,
                        session_income=session_income,
                        price_ivt=price_ivt
                        ).to(device)

In [51]:
df1 = df[df['choice_made'] == 1 ]

In [52]:
df1['ils_udb_yds'].describe()

count     4294.000000
mean      1873.212760
std       1086.537606
min        330.000000
25%       1159.710000
50%       1637.960000
75%       2381.257500
max      13246.500000
Name: ils_udb_yds, dtype: float64

In [53]:
df1['log_l'].describe()

count    4294.000000
mean        3.731750
std         0.288860
min         1.609438
25%         3.688879
50%         3.688879
75%         3.806662
max         4.382027
Name: log_l, dtype: float64

In [55]:
import pandas as pd
import biogeme.database as db
import biogeme.biogeme as bio
from biogeme import models
from biogeme.expressions import Beta, DefineVariable

# Assuming df is your DataFrame already loaded and ready
database = db.Database("MyDiscreteChoiceData", df)


ImportError: cannot import name 'DefineVariable' from 'biogeme.expressions' (C:\Users\hisham\AppData\Roaming\Python\Python311\site-packages\biogeme\expressions\__init__.py)